# Stacking Ensemble Regressor (LightGBM & XGBoost) - Alpha 5D

Notebook này dùng để huấn luyện mô hình **Stacking Ensemble** kết hợp LightGBM và XGBoost thông qua Ridge Regressor (Meta-Learner) trên nền tảng **Google Colab**.

**Quy trình 2 tầng (Stacking):**
- **Tầng 1 (Base Learners):** LightGBM Regressor và XGBoost Regressor học các mối quan hệ phi tuyến từ các đặc trưng kỹ thuật.
- **Tầng 2 (Meta-Learner):** Ridge Regression kết hợp dự báo từ tầng 1, triệt tiêu đa cộng tuyến (multicollinearity) để tối ưu hóa điểm số xếp hạng cuối cùng.

**Mục tiêu dự báo (Target):**
- `alpha_rank_pct`: Thứ hạng phần trăm của tỷ suất sinh lời vượt trội (`alpha_5d = return_5d_stock - return_5d_vnindex`) của cổ phiếu trong 5 phiên tới.

**Đánh giá:**
- Walk-Forward Validation đo lường **Rank IC** và **IR (Information Ratio)**.
- Đánh giá thực chiến trên tập Test bằng các chỉ số: **Top-k avg_alpha, Top-k alpha_win_rate, Top-k avg_stock_return, Top-k stock_win_rate**.

In [ ]:
# === BƯỚC 1: CÀI ĐẶT THƯ VIỆN & KẾT NỐI GOOGLE DRIVE ===
%pip install -q lightgbm xgboost scikit-learn pandas numpy scipy

from google.colab import drive
drive.mount('/content/drive')

import json
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb

In [ ]:
# === BƯỚC 2: LOAD DATA TỪ GOOGLE DRIVE ===
# Đường dẫn file all_stocks_train.csv lưu trên Google Drive của bạn
DATA_PATH = '/content/drive/MyDrive/all_stocks_train.csv'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu tại: {DATA_PATH}. Vui lòng upload file csv lên Google Drive!")

df = pd.read_csv(DATA_PATH, low_memory=False)
df['trading_date'] = pd.to_datetime(df['trading_date'])
df = df.drop_duplicates(subset=['symbol', 'trading_date'], keep='last')
df = df.sort_values(['symbol', 'trading_date']).reset_index(drop=True)

print(f"Raw data loaded: {len(df):,} dòng | {df['symbol'].nunique()} mã cổ phiếu")

In [ ]:
# === BƯỚC 3: TIỀN XỬ LÝ & LỌC CỔ PHIẾU ===
START_DATE = '2021-01-01'
MIN_VOLUME_SMA20 = 100000

# 3.1. Tính toán các chỉ báo cho VNINDEX
vnindex = (
    df[df['symbol'] == 'VNINDEX'][['trading_date', 'close', 'return_5d']]
    .drop_duplicates(subset=['trading_date'], keep='last')
    .sort_values('trading_date')
    .reset_index(drop=True)
)
vnindex['vnindex_momentum_5'] = vnindex['close'].pct_change(5).clip(-0.3, 0.3)
vnindex['vnindex_momentum_20'] = vnindex['close'].pct_change(20).clip(-0.3, 0.3)
vnindex_diff = vnindex['close'].diff()
gain = vnindex_diff.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
loss = (-vnindex_diff.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
rs = gain / loss.replace(0, np.nan)
vnindex['vnindex_rsi'] = 100 - (100 / (1 + rs))
vnindex = vnindex.rename(columns={'return_5d': 'vnindex_return_5d'})

# 3.2. Lọc các cổ phiếu thường (loại bỏ VNINDEX, sàn HOSE/HNX, lọc thanh khoản)
stocks = df[df['symbol'] != 'VNINDEX'].copy()
stocks = stocks[stocks['exchange'].isin(['HOSE', 'HNX'])]
stocks = stocks[stocks['volume_sma_20'] > MIN_VOLUME_SMA20]
stocks = stocks[stocks['trading_date'] >= START_DATE]

# 3.3. Merge thông tin VNINDEX để tính toán chỉ báo tương đối
stocks = stocks.merge(vnindex, on='trading_date', how='left', validate='many_to_one')
stocks['alpha_5d'] = stocks['return_5d'] - stocks['vnindex_return_5d']

# 3.4. Tính toán 3 đặc trưng tương đối
stocks['rel_momentum_5'] = (stocks['price_momentum_5'] - stocks['vnindex_momentum_5']).clip(-0.5, 0.5)
stocks['rel_momentum_20'] = (stocks['price_momentum_20'] - stocks['vnindex_momentum_20']).clip(-0.5, 0.5)
stocks['rsi_vs_vnindex'] = (stocks['rsi_14'] - stocks['vnindex_rsi']).clip(-100, 100)

# Danh sách 10 đặc trưng chính của mô hình
FEATURES = [
    'price_vs_sma5', 'price_vs_sma20',
    'rel_momentum_5', 'rel_momentum_20',
    'rsi_14', 'rsi_vs_vnindex',
    'volume_ratio', 'bb_width_norm',
    'cmf_20', 'atr_pct'
]

# 3.5. Tính toán mục tiêu (Target) alpha_rank_pct
stocks = stocks.dropna(subset=['alpha_5d'])
stocks['alpha_rank_pct'] = stocks.groupby('trading_date')['alpha_5d'].rank(pct=True, method='first')

# Xử lý giá trị trống (forward fill theo mã)
stocks[FEATURES] = stocks.groupby('symbol')[FEATURES].ffill()
stocks = stocks.dropna(subset=FEATURES)

print(f"Tiền xử lý hoàn tất!")
print(f"Số lượng bản ghi khả dụng: {len(stocks):,}")
print(f"Số lượng cổ phiếu: {stocks['symbol'].nunique()}")
print(f"Mục tiêu dự báo (Target): alpha_rank_pct")
print(f"10 Đặc trưng (Features): {FEATURES}")

In [ ]:
# === BƯỚC 4: ĐÁNH GIÁ WALK-FORWARD VALIDATION (STACKING ENSEMBLE) ===
# Thực hiện chia dữ liệu cuốn chiếu trên 5 Folds để đánh giá tính ổn định của Stacking.

dates = sorted(stocks['trading_date'].unique())
n_days = len(dates)

fold_test_days = 40   # 2 tháng kiểm thử
fold_train_days = 360 # 18 tháng huấn luyện
folds = []

for i in range(5):
    test_end_idx = n_days - 1 - (i * fold_test_days)
    test_start_idx = test_end_idx - fold_test_days + 1
    train_end_idx = test_start_idx - 1
    train_start_idx = max(0, train_end_idx - fold_train_days + 1)
    
    folds.append({
        'fold': i + 1,
        'train_dates': dates[train_start_idx:train_end_idx + 1],
        'test_dates': dates[test_start_idx:test_end_idx + 1]
    })

folds = folds[::-1]
ic_records = []

print("BẮT ĐẦU WALK-FORWARD ENSEMBLE BACKTESTING...\n")

for f in folds:
    # Lấy dữ liệu fold hiện tại
    train_fold = stocks[stocks['trading_date'].isin(f['train_dates'])].copy()
    test_fold = stocks[stocks['trading_date'].isin(f['test_dates'])].copy()
    
    # Chia train_fold làm 2 phần: Sub-train (80%) cho tầng 1, Sub-val (20%) cho Ridge
    fold_dates = sorted(train_fold['trading_date'].unique())
    split_date_idx = int(len(fold_dates) * 0.8)
    sub_train_dates = fold_dates[:split_date_idx]
    sub_val_dates = fold_dates[split_date_idx:]
    
    sub_train = train_fold[train_fold['trading_date'].isin(sub_train_dates)]
    sub_val = train_fold[train_fold['trading_date'].isin(sub_val_dates)]
    
    # Tách X, y
    X_sub_tr, y_sub_tr = sub_train[FEATURES], sub_train['alpha_rank_pct']
    X_sub_val, y_sub_val = sub_val[FEATURES], sub_val['alpha_rank_pct']
    X_te, y_te = test_fold[FEATURES], test_fold['alpha_rank_pct']
    
    # Huấn luyện Tầng 1: LightGBM
    lgb_model = lgb.LGBMRegressor(
        n_estimators=100, learning_rate=0.03, max_depth=3, num_leaves=8,
        min_child_samples=100, subsample=0.7, colsample_bytree=0.5,
        random_state=42, n_jobs=-1, verbose=-1
    )
    lgb_model.fit(X_sub_tr, y_sub_tr)
    
    # Huấn luyện Tầng 1: XGBoost
    xgb_model = xgb.XGBRegressor(
        n_estimators=100, learning_rate=0.03, max_depth=3,
        subsample=0.7, colsample_bytree=0.5,
        random_state=42, n_jobs=-1
    )
    xgb_model.fit(X_sub_tr, y_sub_tr)
    
    # Lấy dự báo trên tập Sub-val để huấn luyện Tầng 2
    lgb_val_preds = lgb_model.predict(X_sub_val)
    xgb_val_preds = xgb_model.predict(X_sub_val)
    
    meta_X_val = np.column_stack((lgb_val_preds, xgb_val_preds))
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(meta_X_val, y_sub_val)
    
    # Dự báo trên tập Test Fold
    lgb_test_preds = lgb_model.predict(X_te)
    xgb_test_preds = xgb_model.predict(X_te)
    meta_X_test = np.column_stack((lgb_test_preds, xgb_test_preds))
    
    test_fold['pred_score'] = meta_model.predict(meta_X_test)
    
    # Tính Rank IC từng ngày trên tập test
    daily_ics = []
    for date, group in test_fold.groupby('trading_date'):
        if len(group) < 10:
            continue
        ic, _ = spearmanr(group['pred_score'], group['alpha_5d'])
        if not np.isnan(ic):
            daily_ics.append(ic)
            
    mean_ic = np.mean(daily_ics) if daily_ics else 0
    std_ic = np.std(daily_ics) if daily_ics else 0
    ir = mean_ic / std_ic if std_ic > 0 else 0
    
    ic_records.append({
        'Fold': f['fold'],
        'Train Dates': f"{f['train_dates'][0].strftime('%Y-%m-%d')} -> {f['train_dates'][-1].strftime('%Y-%m-%d')}",
        'Test Dates': f"{f['test_dates'][0].strftime('%Y-%m-%d')} -> {f['test_dates'][-1].strftime('%Y-%m-%d')}",
        'LGB_Weight': meta_model.coef_[0],
        'XGB_Weight': meta_model.coef_[1],
        'Mean Rank IC': mean_ic,
        'Std Rank IC': std_ic,
        'IR': ir
    })
    print(f"Fold {f['fold']} hoàn tất | Trọng số: LGB={meta_model.coef_[0]:.4f}, XGB={meta_model.coef_[1]:.4f} | Rank IC: {mean_ic:.4f} | IR: {ir:.4f}")

df_ic = pd.DataFrame(ic_records)
print("\n" + "="*80)
print("BÁO CÁO WALK-FORWARD STACKING ENSEMBLE VALIDATION")
print("="*80)
print(df_ic[['Fold', 'LGB_Weight', 'XGB_Weight', 'Mean Rank IC', 'Std Rank IC', 'IR']].to_string(index=False))
print(f"\nTrung bình Rank IC trên 5 Folds: {df_ic['Mean Rank IC'].mean():.4f}")
print(f"Trung bình IR (Information Ratio): {df_ic['IR'].mean():.4f}")

In [ ]:
# === BƯỚC 5: HUẤN LUYỆN TOÀN BỘ MÔ HÌNH PRODUCTION ===
# Cắt dữ liệu: Toàn bộ dữ liệu trước 2025-10-01 làm tập huấn luyện, sau đó làm tập Test đánh giá thực tế.
TEST_CUTOFF = pd.Timestamp('2025-10-01')

train_full = stocks[stocks['trading_date'] < TEST_CUTOFF].copy()
test_full = stocks[stocks['trading_date'] >= TEST_CUTOFF].copy()

# Chia nhỏ tập train_full thành 80% train cho base-models, 20% validation cho Ridge
train_dates = sorted(train_full['trading_date'].unique())
split_idx = int(len(train_dates) * 0.8)
sub_train_dates = train_dates[:split_idx]
sub_val_dates = train_dates[split_idx:]

sub_train = train_full[train_full['trading_date'].isin(sub_train_dates)]
sub_val = train_full[train_full['trading_date'].isin(sub_val_dates)]

X_sub_tr, y_sub_tr = sub_train[FEATURES], sub_train['alpha_rank_pct']
X_sub_val, y_sub_val = sub_val[FEATURES], sub_val['alpha_rank_pct']
X_te_full, y_te_full = test_full[FEATURES], test_full['alpha_rank_pct']

print("Đang huấn luyện mô hình Production trên toàn bộ dữ liệu lịch sử...")

# 5.1. Train LGBM ở Tầng 1
model_lgb_prod = lgb.LGBMRegressor(
    n_estimators=100, learning_rate=0.03, max_depth=3, num_leaves=8,
    min_child_samples=100, subsample=0.7, colsample_bytree=0.5,
    random_state=42, n_jobs=-1, verbose=-1
)
model_lgb_prod.fit(X_sub_tr, y_sub_tr)

# 5.2. Train XGBoost ở Tầng 1
model_xgb_prod = xgb.XGBRegressor(
    n_estimators=100, learning_rate=0.03, max_depth=3,
    subsample=0.7, colsample_bytree=0.5,
    random_state=42, n_jobs=-1
)
model_xgb_prod.fit(X_sub_tr, y_sub_tr)

# 5.3. Predict trên Sub-val để train Ridge ở Tầng 2
lgb_val_preds = model_lgb_prod.predict(X_sub_val)
xgb_val_preds = model_xgb_prod.predict(X_sub_val)
meta_X_val = np.column_stack((lgb_val_preds, xgb_val_preds))

model_meta_prod = Ridge(alpha=1.0)
model_meta_prod.fit(meta_X_val, y_sub_val)

print(f"\n[Thành công] Mô hình Stacking đã học xong!")
print(f"Trọng số Meta-Learner cuối cùng: LightGBM = {model_meta_prod.coef_[0]:.4f} | XGBoost = {model_meta_prod.coef_[1]:.4f}")

In [ ]:
# === BƯỚC 6: ĐÁNH GIÁ THỰC CHIẾN TẬP TEST (CÓ VÀ KHÔNG CÓ BỘ LỌC MARKET REGIME) ===
# Chạy dự đoán cho toàn bộ tập Test
lgb_test_preds = model_lgb_prod.predict(X_te_full)
xgb_test_preds = model_xgb_prod.predict(X_te_full)
meta_X_test = np.column_stack((lgb_test_preds, xgb_test_preds))

# Copy các cột dữ liệu cần thiết bao gồm các chỉ báo VNINDEX để làm bộ lọc
test_eval = test_full[['symbol', 'trading_date', 'alpha_5d', 'return_5d', 'vnindex_return_5d', 'vnindex_momentum_20', 'vnindex_rsi']].copy()
test_eval['score'] = model_meta_prod.predict(meta_X_test)

# Định nghĩa hàm tính toán các chỉ số Top-k
def evaluate_topk(frame, ks=(5, 10, 20, 50), ascending=False):
    rows = []
    for k in ks:
        picks = (
            frame.sort_values(['trading_date', 'score'], ascending=[True, ascending])
                 .groupby('trading_date', group_keys=False)
                 .head(k)
        )
        rows.append({
            'top_k': k,
            'samples': len(picks),
            'days': picks['trading_date'].nunique(),
            'avg_alpha': picks['alpha_5d'].mean(),
            'alpha_win_rate': (picks['alpha_5d'] > 0).mean(),
            'avg_stock_return': picks['return_5d'].mean(),
            'stock_win_rate': (picks['return_5d'] > 0).mean(),
        })
    return pd.DataFrame(rows)

# --- BÁO CÁO 1: CHƯA CÓ BỘ LỌC PHÒNG VỆ (RAW PERFORMANCE) ---
test_topk = evaluate_topk(test_eval, ks=(5, 10, 20, 50), ascending=False)
test_bottomk = evaluate_topk(test_eval, ks=(5, 10, 20, 50), ascending=True)
universe_test = test_full.groupby('trading_date')[['alpha_5d', 'return_5d']].mean().mean()

print('=' * 80)
print('BÁO CÁO 1: HIỆU SUẤT THÔ CỦA AI (CHƯA CÓ BỘ LỌC PHÒNG VỆ MARKET REGIME)')
print('=' * 80)
print(f"Test average daily alpha universe  : {universe_test['alpha_5d']:.4%}")
print(f"Test average daily return universe : {universe_test['return_5d']:.4%}")

print('\n[MUA - RAW] Top-k cổ phiếu AI khuyến nghị cao nhất:')
print(test_topk.to_string(index=False))

print('\n[BÁN - RAW] Top-k cổ phiếu AI khuyến nghị thấp nhất:')
print(test_bottomk.to_string(index=False))

test_ics = []
for date, group in test_eval.groupby('trading_date'):
    if len(group) < 10:
        continue
    ic, _ = spearmanr(group['score'], group['alpha_5d'])
    if not np.isnan(ic):
        test_ics.append(ic)
print(f"\nTrung bình Spearman Rank IC (Raw): {np.mean(test_ics):.4f}")


# --- BÁO CÁO 2: ĐÃ CÓ BỘ LỌC PHÒNG VỆ (TRUE DEFENSIVE PERFORMANCE) ---
# Điều kiện kích hoạt rủi ro thị trường (giống daily_predict.py):
# skip giao dịch nếu vnindex_momentum_20 < -0.02 HOẶC vnindex_rsi < 45
market_active_mask = (test_eval['vnindex_momentum_20'] >= -0.02) & (test_eval['vnindex_rsi'] >= 45)
test_eval_filtered = test_eval[market_active_mask].copy()

total_days = test_eval['trading_date'].nunique()
active_days = test_eval_filtered['trading_date'].nunique()
skipped_days = total_days - active_days

print('\n' + '=' * 80)
print('BÁO CÁO 2: HIỆU SUẤT THỰC CHIẾN KHI ÁP DỤNG MARKET REGIME FILTER')
print('=' * 80)
print(f"Tổng số ngày trong chu kỳ kiểm thử: {total_days} ngày")
print(f"Số ngày KÍCH HOẠT PHÒNG THỦ (Skip giao dịch, giữ tiền mặt)  : {skipped_days} ngày ({skipped_days/total_days:.2%})")
print(f"Số ngày GIAO DỊCH THỰC TẾ (Active Trading Days)             : {active_days} ngày ({active_days/total_days:.2%})")

if active_days > 0:
    test_topk_filtered = evaluate_topk(test_eval_filtered, ks=(5, 10, 20, 50), ascending=False)
    test_bottomk_filtered = evaluate_topk(test_eval_filtered, ks=(5, 10, 20, 50), ascending=True)
    
    # Tính hiệu suất trung bình của thị trường chỉ trong những ngày giao dịch active
    active_dates_list = test_eval_filtered['trading_date'].unique()
    universe_active = test_full[test_full['trading_date'].isin(active_dates_list)].groupby('trading_date')[['alpha_5d', 'return_5d']].mean().mean()
    
    print(f"\n[Chỉ tính ngày Active] average daily alpha universe  : {universe_active['alpha_5d']:.4%}")
    print(f"[Chỉ tính ngày Active] average daily return universe : {universe_active['return_5d']:.4%}")
    
    print('\n[MUA - FILTERED] Top-k cổ phiếu AI khuyến nghị cao nhất:')
    print(test_topk_filtered.to_string(index=False))
    
    print('\n[BÁN - FILTERED] Top-k cổ phiếu AI khuyến nghị thấp nhất:')
    print(test_bottomk_filtered.to_string(index=False))
    
    test_ics_filtered = []
    for date, group in test_eval_filtered.groupby('trading_date'):
        if len(group) < 10:
            continue
        ic, _ = spearmanr(group['score'], group['alpha_5d'])
        if not np.isnan(ic):
            test_ics_filtered.append(ic)
    print(f"\nTrung bình Spearman Rank IC (Filtered - Ngày active): {np.mean(test_ics_filtered):.4f}")
else:
    print("\n[CẢNH BÁO] Toàn bộ chu kỳ test đều kích hoạt phòng thủ. Không có ngày nào active để giao dịch.")


In [ ]:
# === BƯỚC 7: XUẤT MODEL VÀ CONFIG SANG GOOGLE DRIVE ===
OUTPUT_DIR = '/content/drive/MyDrive/StockData/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = f'{OUTPUT_DIR}/ensemble_model.pkl'
CONFIG_PATH = f'{OUTPUT_DIR}/ensemble_features.json'

# Lưu cả 3 thành phần của mô hình Stacking vào file pkl
with open(MODEL_PATH, 'wb') as f:
    pickle.dump({
        'lgbm': model_lgb_prod,
        'xgb': model_xgb_prod,
        'meta': model_meta_prod
    }, f)

# Lưu file config mô hình
with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump({
        'model_type': 'stacking_ensemble',
        'target': 'alpha_rank_pct',
        'features': FEATURES
    }, f, indent=2, ensure_ascii=False)

print("Đã lưu các file mô hình thành công vào Google Drive!")
print(f"Model path  : {MODEL_PATH}")
print(f"Config path : {CONFIG_PATH}")
print("-> Hãy tải file 'ensemble_model.pkl' và 'ensemble_features.json' về đặt vào thư mục dự án ở Local để chạy daily_predict!")